DATASET1 İÇİN,DİĞER ML DEMOLARIYLA AYNI MANTIKTA METRİKLERİ,TEK TAHMİNİ VE KLASÖRÜ GÖRECEĞİZ

In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
%run MLProject.ipynb

# Dataset 1 hazırlığı
df = pd.read_csv('usgs_main.csv')
df1 = df.copy()
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1.sort_values('time')
df1 = df1.dropna(subset=['latitude','longitude','depth','mag','time'])

dfweek_mlp = df1.set_index('time').resample('W').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
    'magType': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'status': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'net': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
})

dfweek_mlp = dfweek_mlp.reset_index(drop=True)
dfweek_mlp.index = dfweek_mlp.index + 1
dfweek_mlp.index.name = 'timeindex'

dfweek_mlp['futuremag'] = dfweek_mlp['mag'].shift(-1)
dfweek_mlp['futuredepth'] = dfweek_mlp['depth'].shift(-1)
dfweek_mlp['futurelat'] = dfweek_mlp['latitude'].shift(-1)
dfweek_mlp['futurelon'] = dfweek_mlp['longitude'].shift(-1)
dfweek_mlp = dfweek_mlp.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])

train_mlp = dfweek_mlp[:int((4*len(dfweek_mlp))/5)]
test_mlp = dfweek_mlp[int(4*len(dfweek_mlp)/5):]


In [2]:
best_model_mlp_1 = joblib.load('models/mlp_dataset1.pkl')
test_predictions_mlp = best_model_mlp_1.predict(test_mlp)
test_actual_mlp = test_mlp[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len_mlp = min(len(test_predictions_mlp), len(test_actual_mlp))
target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']

In [3]:
print("DATASET 1 MLP SONUÇLARI:")
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_mlp.shape[1] and i < test_actual_mlp.shape[1]:
        target_mse = mean_squared_error(
            test_actual_mlp.iloc[:min_len_mlp, i], 
            test_predictions_mlp[:min_len_mlp, i]
        )
        target_mae = mean_absolute_error(
            test_actual_mlp.iloc[:min_len_mlp, i], 
            test_predictions_mlp[:min_len_mlp, i]
        )
        target_r2 = r2_score(
            test_actual_mlp.iloc[:min_len_mlp, i], 
            test_predictions_mlp[:min_len_mlp, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

DATASET 1 MLP SONUÇLARI:
   Magnitude   : MSE=0.016, MAE=0.091, R²=-0.680
   Depth       : MSE=8.188, MAE=2.371, R²=-0.355
   Latitude    : MSE=2.200, MAE=1.202, R²=-0.428
   Longitude   : MSE=40.376, MAE=5.353, R²=-1.199


In [4]:
magnitude_thresholds = [2.0, 2.5, 3.0, 3.5, 4.0]

for threshold in magnitude_thresholds:
    predicted_magnitudes = test_predictions_mlp[:min_len_mlp, 0]
    actual_magnitudes = test_actual_mlp.iloc[:min_len_mlp, 0].values
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"\nBüyüklük Eşiği: {threshold}")
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=2.0): 0
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.5
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=3.5): 0
Tahmin edilen deprem sayısı (>=3.5): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 4.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=4.0): 0
Tahmin edilen deprem sayısı (>=4.0): 0
Doğru tahmin sayısı: 

c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct s

KONUM KOD BLOĞU

In [5]:
test_with_predictions_mlp = test_mlp.iloc[:min_len_mlp].copy()
test_with_predictions_mlp['predicted_mag'] = test_predictions_mlp[:min_len_mlp, 0]
test_with_predictions_mlp['predicted_lat'] = test_predictions_mlp[:min_len_mlp, 2]
test_with_predictions_mlp['predicted_lon'] = test_predictions_mlp[:min_len_mlp, 3]

test_with_predictions_mlp['lat_group'] = np.round(test_with_predictions_mlp['latitude'])
test_with_predictions_mlp['lon_group'] = np.round(test_with_predictions_mlp['longitude'])
test_with_predictions_mlp['location_group'] = test_with_predictions_mlp['lat_group'].astype(str) + '_' + test_with_predictions_mlp['lon_group'].astype(str)

location_groups_mlp = test_with_predictions_mlp.groupby('location_group').size()
valid_locations_mlp = location_groups_mlp[location_groups_mlp >= 1].index

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_mlp)}")

threshold = 1.5

for location in valid_locations_mlp[:10]:
    location_data = test_with_predictions_mlp[test_with_predictions_mlp['location_group'] == location]
    lat, lon = location.split('_')
    
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()


Yeterli veri olan bölge sayısı: 7
Bölge (37.0°, -108.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.78
  Max tahmin büyüklük: 1.72

Bölge (37.0°, -112.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.60
  Max tahmin büyüklük: 1.78

Bölge (37.0°, -114.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.66
  Max tahmin büyüklük: 1.67

Bölge (37.0°, -115.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.68
  Max tahmin büyüklük: 1.69

Bölge (37.0°, -117.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.85
  Max tahmin büyüklük: 1.71

Bölge (38.0°, -113.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.72
  Max tahmin büyüklük: 1.73

Bölge (40.0°, -118.0°) -

DATASET2

In [6]:
pn = pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
df2 = pn.copy()
df2 = df2.rename(columns={
    'Time':'time', 'Mag':'mag', 'Depth':'depth', 
    'Latitude':'latitude', 'Longitude':'longitude'
})

df2['time'] = pd.to_datetime(df2['time'])
df2 = df2.sort_values('time')
df2 = df2.dropna(subset=['latitude','longitude','depth','mag','time'])

dfyear_mlp = df2.set_index('time').resample('YE').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
    'MagType': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown',
    'Type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown',
    'status': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown',
    'net': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown'
})

dfyear_mlp = dfyear_mlp.rename(columns={'MagType':'magType','Type':'type'})

dfyear_mlp = dfyear_mlp.reset_index(drop=True)
dfyear_mlp.index = dfyear_mlp.index + 1
dfyear_mlp.index.name = 'timeindex'

dfyear_mlp['futuremag'] = dfyear_mlp['mag'].shift(-1)
dfyear_mlp['futuredepth'] = dfyear_mlp['depth'].shift(-1)
dfyear_mlp['futurelat'] = dfyear_mlp['latitude'].shift(-1)
dfyear_mlp['futurelon'] = dfyear_mlp['longitude'].shift(-1)
dfyear_mlp = dfyear_mlp.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])

train_mlp_2 = dfyear_mlp[:int((4*len(dfyear_mlp))/5)]
test_mlp_2 = dfyear_mlp[int(4*len(dfyear_mlp)/5):]


In [7]:
best_model_mlp_2 = joblib.load('models/mlp_dataset2.pkl')
test_predictions_mlp_2 = best_model_mlp_2.predict(test_mlp_2)
test_actual_mlp_2 = test_mlp_2[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len_mlp_2 = min(len(test_predictions_mlp_2), len(test_actual_mlp_2))


DATASET 2 İÇİN 3 FARKLI PERFORMANS ÖLÇÜMÜ

In [8]:
print("\nDATASET 2 MLP SONUÇLARI:")
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_mlp_2.shape[1] and i < test_actual_mlp_2.shape[1]:
        target_mse = mean_squared_error(
            test_actual_mlp_2.iloc[:min_len_mlp_2, i], 
            test_predictions_mlp_2[:min_len_mlp_2, i]
        )
        target_mae = mean_absolute_error(
            test_actual_mlp_2.iloc[:min_len_mlp_2, i], 
            test_predictions_mlp_2[:min_len_mlp_2, i]
        )
        target_r2 = r2_score(
            test_actual_mlp_2.iloc[:min_len_mlp_2, i], 
            test_predictions_mlp_2[:min_len_mlp_2, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")


DATASET 2 MLP SONUÇLARI:
   Magnitude   : MSE=0.003, MAE=0.046, R²=-10.041
   Depth       : MSE=83.883, MAE=7.526, R²=0.257
   Latitude    : MSE=21.617, MAE=3.830, R²=-0.853
   Longitude   : MSE=567.805, MAE=19.930, R²=-2.208


In [9]:
magnitude_thresholds_2 = [6.0, 6.5, 7.0, 7.5, 8.0]

for threshold in magnitude_thresholds_2:
    predicted_magnitudes = test_predictions_mlp_2[:min_len_mlp_2, 0]
    actual_magnitudes = test_actual_mlp_2.iloc[:min_len_mlp_2, 0].values
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"\nBüyüklük Eşiği: {threshold}")
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")



Büyüklük Eşiği: 6.0
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=6.0): 0
Tahmin edilen deprem sayısı (>=6.0): 1
Doğru tahmin sayısı: 20
Doğruluk (Accuracy): 0.952
Kesinlik (Precision): 0.000
Duyarlılık (Recall): 0.000
F1-Score: 0.000
Confusion Matrix: TN=20, FP=1, FN=0, TP=0

Büyüklük Eşiği: 6.5
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=6.5): 0
Tahmin edilen deprem sayısı (>=6.5): 0
Doğru tahmin sayısı: 21
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.0
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=7.0): 0
Tahmin edilen deprem sayısı (>=7.0): 0
Doğru tahmin sayısı: 21
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.5
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=7.5): 0
Tahmin edilen deprem sayısı (>=7.5): 0
Doğru tahmin sayısı: 21
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 8.0
Toplam örnek sayısı: 21
Gerçek deprem sayısı 

c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct s

In [10]:
print("\nKONUM KONTROLÜ - DATASET 2:")
test_with_predictions_mlp_2 = test_mlp_2.iloc[:min_len_mlp_2].copy()
test_with_predictions_mlp_2['predicted_mag'] = test_predictions_mlp_2[:min_len_mlp_2, 0]
test_with_predictions_mlp_2['predicted_lat'] = test_predictions_mlp_2[:min_len_mlp_2, 2]
test_with_predictions_mlp_2['predicted_lon'] = test_predictions_mlp_2[:min_len_mlp_2, 3]

test_with_predictions_mlp_2['lat_group'] = np.round(test_with_predictions_mlp_2['latitude'] / 5.0) * 5.0
test_with_predictions_mlp_2['lon_group'] = np.round(test_with_predictions_mlp_2['longitude'] / 5.0) * 5.0
test_with_predictions_mlp_2['location_group'] = test_with_predictions_mlp_2['lat_group'].astype(str) + '_' + test_with_predictions_mlp_2['lon_group'].astype(str)

location_groups_mlp_2 = test_with_predictions_mlp_2.groupby('location_group').size()
valid_locations_mlp_2 = location_groups_mlp_2[location_groups_mlp_2 >= 1].index

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_mlp_2)}")

threshold = 6.0

for location in valid_locations_mlp_2[:10]:
    location_data = test_with_predictions_mlp_2[test_with_predictions_mlp_2['location_group'] == location]
    lat, lon = location.split('_')
    
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()


KONUM KONTROLÜ - DATASET 2:
Yeterli veri olan bölge sayısı: 17
Bölge (-0.0°, 25.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.87
  Max tahmin büyüklük: 5.94

Bölge (-0.0°, 30.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.90
  Max tahmin büyüklük: 5.98

Bölge (-0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.88

Bölge (-0.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.89
  Max tahmin büyüklük: 5.86

Bölge (-0.0°, 55.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.88
  Max tahmin büyüklük: 5.80

Bölge (-5.0°, 10.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.89
  Max tahmin büyüklük: 5.94

Bölge 

MLP DE EN İYİ SONUÇ ALDIĞIMIZ DATA

In [11]:
dfday_mlp = df1.set_index('time').resample('D').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
    'magType': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'status': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'net': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
})

dfday_mlp = dfday_mlp.reset_index(drop=True)
dfday_mlp.index = dfday_mlp.index + 1
dfday_mlp.index.name = 'timeindex'
numerical_cols = ['mag', 'latitude', 'longitude', 'depth']
outlier_detector = IsolationForest(
    contamination=0.1,
    random_state=42,
    n_estimators=100
)
outlier_mask = outlier_detector.fit_predict(dfday_mlp[numerical_cols]) == 1
dfday_mlp = dfday_mlp[outlier_mask]
print(f"Outlier detection sonrası veri sayısı: {len(dfday_mlp)}")
print(f"Temizlenen outlier sayısı: {np.sum(~outlier_mask)}")

# Robust scaling
robust_scaler = RobustScaler()
robust_cols = ['mag', 'latitude', 'longitude', 'depth']
available_robust_cols = [col for col in robust_cols if col in dfday_mlp.columns]

if available_robust_cols:
    robust_scaled_data = robust_scaler.fit_transform(dfday_mlp[available_robust_cols])
    for i, col in enumerate(available_robust_cols):
        dfday_mlp[f'{col}_robust'] = robust_scaled_data[:, i]
dfday_mlp['futuremag'] = dfday_mlp['mag'].shift(-1)
dfday_mlp['futuredepth'] = dfday_mlp['depth'].shift(-1)
dfday_mlp['futurelat'] = dfday_mlp['latitude'].shift(-1)
dfday_mlp['futurelon'] = dfday_mlp['longitude'].shift(-1)
dfday_mlp = dfday_mlp.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])
trainday_mlp = dfday_mlp[:int((4*len(dfday_mlp))/5)]
testday_mlp = dfday_mlp[int(4*len(dfday_mlp)/5):]


Outlier detection sonrası veri sayısı: 256
Temizlenen outlier sayısı: 29


In [12]:
best_model_mlp_3 = joblib.load('models/bestresult.pkl')
test_predictions_mlp_3 = best_model_mlp_3.predict(testday_mlp)
test_actual_mlp_3 = testday_mlp[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len_mlp_3 = min(len(test_predictions_mlp_3), len(test_actual_mlp_3))

METRİK ÖLÇÜÜMLERİ,YİNE BENZER BLOKLARLA SAĞLAM TAHMİN

In [13]:
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_mlp_3.shape[1] and i < test_actual_mlp_3.shape[1]:
        target_mse = mean_squared_error(
            test_actual_mlp_3.iloc[:min_len_mlp_3, i], 
            test_predictions_mlp_3[:min_len_mlp_3, i]
        )
        target_mae = mean_absolute_error(
            test_actual_mlp_3.iloc[:min_len_mlp_3, i], 
            test_predictions_mlp_3[:min_len_mlp_3, i]
        )
        target_r2 = r2_score(
            test_actual_mlp_3.iloc[:min_len_mlp_3, i], 
            test_predictions_mlp_3[:min_len_mlp_3, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

   Magnitude   : MSE=0.034, MAE=0.145, R²=-0.607
   Depth       : MSE=12.404, MAE=2.756, R²=-0.109
   Latitude    : MSE=2.251, MAE=1.166, R²=0.052
   Longitude   : MSE=28.438, MAE=4.546, R²=-0.216


In [14]:
magnitude_thresholds = [1.5,2.0, 2.5, 3.0, 3.5, 4.0]

for threshold in magnitude_thresholds:
    print(f"\n Büyüklük Eşiği: {threshold}")
    
    # Tahmin edilen ve gerçek büyüklükler
    predicted_magnitudes = test_predictions_mlp_3[:min_len_mlp_3, 0] 
    actual_magnitudes = test_actual_mlp_3.iloc[:min_len_mlp_3, 0].values
    #Threshold a göre 
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    # Detaylı istatistikler
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")  
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


 Büyüklük Eşiği: 1.5
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=1.5): 44
Tahmin edilen deprem sayısı (>=1.5): 47
Doğru tahmin sayısı: 42
Doğruluk (Accuracy): 0.857
Kesinlik (Precision): 0.894
Duyarlılık (Recall): 0.955
F1-Score: 0.923
Confusion Matrix: TN=0, FP=5, FN=2, TP=42

 Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=2.0): 0
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 49
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

 Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 49
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

 Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 49
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

 Büyüklük Eşiği: 3.5
Toplam örnek sayısı: 49
Gerçek deprem 

c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct s

In [15]:

test_with_predictions_3 = testday_mlp.iloc[:min_len_mlp_3].copy()
test_with_predictions_3['predicted_mag'] = test_predictions_mlp_3[:min_len_mlp_3, 0]
test_with_predictions_3['predicted_lat'] = test_predictions_mlp_3[:min_len_mlp_3, 2]
test_with_predictions_3['predicted_lon'] = test_predictions_mlp_3[:min_len_mlp_3, 3]

test_with_predictions_3['lat_group'] = np.round(test_with_predictions_3['latitude'])
test_with_predictions_3['lon_group'] = np.round(test_with_predictions_3['longitude'])
test_with_predictions_3['location_group'] = test_with_predictions_3['lat_group'].astype(str) + '_' + test_with_predictions_3['lon_group'].astype(str)

location_groups = test_with_predictions_3.groupby('location_group').size()
valid_locations = location_groups[location_groups >= 2].index  # En az 2 veri noktası

threshold = 1.70

for location in valid_locations:  #En az 2 veri olan 10 noktaya baktık
    location_data = test_with_predictions_3[test_with_predictions_3['location_group'] == location]
    lat, lon = location.split('_')#yine yapay zeka eklemesi
    
    # Bölgede threshold'dan büyük deprem var mı bakıyoruz
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    
    # Doğru tahmin mi?
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    #Burada görsel olarak güzel gözükmesi için yapay zekaya yazdırdım.
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()

Bölge (36.0°, -120.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.82
  Tahmin büyüklük: 1.67

Bölge (37.0°, -110.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.91
  Tahmin büyüklük: 1.62

Bölge (37.0°, -117.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.67
  Tahmin büyüklük: 1.61

Bölge (38.0°, -111.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.91
  Tahmin büyüklük: 1.63

Bölge (38.0°, -115.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.73
  Tahmin büyüklük: 1.69

Bölge (38.0°, -116.0°) - Veri sayısı: 3
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.70
  Tahmin büyüklük: 1.62

Bölge (39.0°, -112.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerç

DİĞER ML DEMOLARINDAKİ GİBİ TEK VERİ TAHMİNİ

In [16]:
def single_prediction_mlp(test_data, model, sample_index=5):
    if sample_index >= len(test_data):
        return None, None
    
    shiftnum = getattr(model, 'shiftnum', 3)
    start_index = max(0, sample_index - shiftnum)
    end_index = sample_index + 1
    
    if sample_index < shiftnum:
        return None, None
    
    sample_rows = test_data.iloc[start_index:end_index]
    target_row = test_data.iloc[sample_index:sample_index+1]
    
    feature_cols = ['mag', 'depth', 'latitude', 'longitude']
    for col in feature_cols:
        if col in target_row.columns:
            print(f"  {col:12}: {target_row[col].iloc[0]:.4f}")
    
    target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
    for col in target_cols:
        if col in target_row.columns:
            print(f"  {col:12}: {target_row[col].iloc[0]:.4f}")
    
    try:
        prediction = model.predict(sample_rows)
        
        if len(prediction) > 0:
            final_prediction = prediction[-1]
            
            print(f"\n MLP TAHMİNLER VE HATALAR:")
            target_labels = ['Future Mag', 'Future Depth', 'Future Lat', 'Future Lon']
            
            for i, (col, label) in enumerate(zip(target_cols, target_labels)):
                if col in target_row.columns and i < len(final_prediction):
                    actual_val = target_row[col].iloc[0]
                    pred_val = final_prediction[i]
                    error = abs(actual_val - pred_val)
                    
                    print(f"  {label:12}: Tahmin={pred_val:.4f}, Gerçek={actual_val:.4f}, Hata={error:.4f}")
            
            return final_prediction.reshape(1, -1), target_row[target_cols].values
        else:
            print("Model tahmin döndürmedi!")
            return None, None
            
    except Exception as e:
        print(f"Hata: {str(e)}")
        return None, None

print("\nTEK VERİ TAHMİN TESTİ:")
print("DATASET 1 MLP TESTİ:")
tahmin_mlp1, gercek_mlp1 = single_prediction_mlp(test_mlp, best_model_mlp_1, sample_index=5)

print("\nDATASET 2 MLP TESTİ:")
tahmin_mlp2, gercek_mlp2 = single_prediction_mlp(test_mlp_2, best_model_mlp_2, sample_index=5)

print("\n GELİŞTİRİLMİŞ MODEL MLP TESTİ:")
tahmin_mlp3, gercek_mlp3 = single_prediction_mlp(testday_mlp, best_model_mlp_3, sample_index=5)


TEK VERİ TAHMİN TESTİ:
DATASET 1 MLP TESTİ:
  mag         : 1.5968
  depth       : 20.3829
  latitude    : 39.6866
  longitude   : -117.9273
  futuremag   : 1.8904
  futuredepth : 23.0301
  futurelat   : 36.5783
  futurelon   : -108.0801

 MLP TAHMİNLER VE HATALAR:
  Future Mag  : Tahmin=1.7137, Gerçek=1.8904, Hata=0.1767
  Future Depth: Tahmin=22.7257, Gerçek=23.0301, Hata=0.3044
  Future Lat  : Tahmin=38.3670, Gerçek=36.5783, Hata=1.7887
  Future Lon  : Tahmin=-111.9352, Gerçek=-108.0801, Hata=3.8551

DATASET 2 MLP TESTİ:
  mag         : 5.8511
  depth       : 62.5755
  latitude    : -1.2222
  longitude   : 39.0798
  futuremag   : 5.8563
  futuredepth : 57.3217
  futurelat   : 1.0368
  futurelon   : 46.4125

 MLP TAHMİNLER VE HATALAR:
  Future Mag  : Tahmin=5.9385, Gerçek=5.8563, Hata=0.0822
  Future Depth: Tahmin=78.4222, Gerçek=57.3217, Hata=21.1006
  Future Lat  : Tahmin=-0.5781, Gerçek=1.0368, Hata=1.6150
  Future Lon  : Tahmin=27.5948, Gerçek=46.4125, Hata=18.8177

 GELİŞTİRİLM

KLASÖR OKUMA,DİĞER ML MODELLERİYLE AYNI MANTIK

In [17]:
import os
import glob

def create_sample_folder(test_data, folder_name="demo_samples", n_samples=15, shiftnum=3):
    os.makedirs(folder_name, exist_ok=True)
    
    print(f"{folder_name} klasörü oluşturuluyor...")
    
    max_start_idx = len(test_data) - shiftnum - 1
    np.random.seed(42)
    sample_count = min(n_samples, max_start_idx)
    start_indices = np.random.choice(max_start_idx, size=sample_count, replace=False)
    
    for i, start_idx in enumerate(start_indices):
        end_idx = start_idx + shiftnum + 1
        sample_chunk = test_data.iloc[start_idx:end_idx]
        
        filename = f"{folder_name}/sample_{i+1:02d}.csv"
        sample_chunk.to_csv(filename, index=False)
    
    return folder_name

def predict_from_folder(folder_name, model, shiftnum=3):
   #tüm csv ler için tahmin yapar
    csv_files = glob.glob(f"{folder_name}/*.csv")
    csv_files.sort()
    
    all_predictions = []
    all_actuals = []
    all_filenames = []
    
    print(f"\n{folder_name} klasöründeki {len(csv_files)} dosya işleniyor...")
    
    for csv_file in csv_files:
        try:
            sample_df = pd.read_csv(csv_file)
            if len(sample_df) < shiftnum + 1:
                continue
                
            target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
            available_targets = [col for col in target_cols if col in sample_df.columns]
            if len(available_targets) == 0:
                continue
            
            actual_values = sample_df[available_targets].iloc[-1].values
            
            try:
                prediction = model.predict(sample_df)
                
                if len(prediction) > 0:
                    final_prediction = prediction[-1]
                    all_predictions.append(final_prediction)
                    all_actuals.append(actual_values)
                    all_filenames.append(os.path.basename(csv_file))
                else:
                    print(f"{os.path.basename(csv_file)}: Model tahmin döndürmedi")
                    
            except Exception as pred_error:
                print(f"{os.path.basename(csv_file)}: Tahmin hatası - {str(pred_error)}")
            
        except Exception as e:
            print(f"{csv_file}: Dosya okuma hatası - {str(e)}")
    
    if len(all_predictions) > 0:
        return np.array(all_predictions), np.array(all_actuals), all_filenames
    else:
        return np.array([]), np.array([]), []

def run_demo(test_data, model, model_name="Model", n_samples=15):
   #demo testi
    shiftnum = getattr(model, 'shiftnum', 3)
    
    folder_name = f"demo_samples_{model_name.lower().replace(' ', '_')}"
    create_sample_folder(test_data, folder_name, n_samples, shiftnum)
    
    predictions, actuals, filenames = predict_from_folder(folder_name, model, shiftnum)
    
    if len(predictions) > 0:
        target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
        
        for i, label in enumerate(target_labels):
            if i < predictions.shape[1] and i < actuals.shape[1]:
                mse = mean_squared_error(actuals[:, i], predictions[:, i])
                mae = mean_absolute_error(actuals[:, i], predictions[:, i])
                r2 = r2_score(actuals[:, i], predictions[:, i])
                
                print(f"{label:<12}: MSE={mse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")
    
    return predictions, actuals, filenames

def predict_single_file(file_path, model):
    #tek bir veri için tahmin
    try:
        sample_df = pd.read_csv(file_path)
        shiftnum = getattr(model, 'shiftnum', 3)#boyut hatası önlemi için eklendi
        
        if len(sample_df) < shiftnum + 1:
            return None, None
        
        # Son satırdaki mevcut değerleri göster
        last_row = sample_df.iloc[-1]
        print(f"  Mevcut Büyüklük: {last_row['mag']:.3f}")
        print(f"  Mevcut Derinlik: {last_row['depth']:.3f}")
        print(f"  Mevcut Konum: ({last_row['latitude']:.3f}, {last_row['longitude']:.3f})")
        
        target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
        available_targets = [col for col in target_cols if col in sample_df.columns]
        
        if len(available_targets) == 0:
            return None, None
        
        print(f"\nHEDEF DEĞERLER:")
        actual_values = []
        for col in available_targets:
            val = last_row[col]
            actual_values.append(val)
            col_name = col.replace('future', '').title()
            print(f"  Gelecek {col_name}: {val:.3f}")
        
        
        prediction = model.predict(sample_df)
        
        if len(prediction) > 0:
            final_prediction = prediction[-1]
            
            print(f"\nMODEL TAHMİNLERİ:")
            target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
            
            for i, (col, label) in enumerate(zip(available_targets, target_labels)):
                if i < len(final_prediction) and i < len(actual_values):
                    pred_val = final_prediction[i]
                    actual_val = actual_values[i]
                    error = abs(actual_val - pred_val)
                    
                    print(f"  {label:<12}: Tahmin={pred_val:.3f}, Gerçek={actual_val:.3f}, Hata={error:.3f}")
            
            return final_prediction, np.array(actual_values)
        else:
            return None, None
            
    except Exception as e:
        print(f"Hata: {str(e)}")
        return None, None


hepsi için tüm klasörü ve tek veriyi okuyan testler

In [18]:
pred_mlp1, act_mlp1, files_mlp1 = run_demo(
    test_mlp, best_model_mlp_1, "MLP_Dataset1", 10
)
sample_file = "demo_samples_mlp_dataset1/sample_01.csv"
if os.path.exists(sample_file):
    pred_single_mlp, act_single_mlp = predict_single_file(sample_file, best_model_mlp_1)
else:
    print(f"Örnek dosya bulunamadı: {sample_file}")

pred_mlp2, act_mlp2, files_mlp2 = run_demo(
    test_mlp_2, best_model_mlp_2, "MLP_Dataset2", 8
)

sample_file_2 = "demo_samples_mlp_dataset2/sample_01.csv"
if os.path.exists(sample_file_2):
    pred_single_mlp2, act_single_mlp2 = predict_single_file(sample_file_2, best_model_mlp_2)
else:
    print(f" Örnek dosya bulunamadı: {sample_file_2}")


pred_mlp3, act_mlp3, files_mlp3 = run_demo(
    testday_mlp, best_model_mlp_3, "MLP_BestResult", 12
)


sample_file_3 = "demo_samples_mlp_bestresult/sample_01.csv"
if os.path.exists(sample_file_3):
    pred_single_mlp3, act_single_mlp3 = predict_single_file(sample_file_3, best_model_mlp_3)
else:
    print(f" Örnek dosya bulunamadı: {sample_file_3}")

demo_samples_mlp_dataset1 klasörü oluşturuluyor...

demo_samples_mlp_dataset1 klasöründeki 6 dosya işleniyor...
Magnitude   : MSE=0.011, MAE=0.085, R²=-0.033
Depth       : MSE=7.165, MAE=2.056, R²=-0.007
Latitude    : MSE=2.434, MAE=1.377, R²=-0.401
Longitude   : MSE=41.970, MAE=5.235, R²=-1.063
  Mevcut Büyüklük: 1.720
  Mevcut Derinlik: 21.887
  Mevcut Konum: (37.446, -113.507)

HEDEF DEĞERLER:
  Gelecek Mag: 1.659
  Gelecek Depth: 18.975
  Gelecek Lat: 37.308
  Gelecek Lon: -116.806

MODEL TAHMİNLERİ:
  Magnitude   : Tahmin=1.691, Gerçek=1.659, Hata=0.033
  Depth       : Tahmin=20.940, Gerçek=18.975, Hata=1.964
  Latitude    : Tahmin=38.167, Gerçek=37.308, Hata=0.859
  Longitude   : Tahmin=-110.053, Gerçek=-116.806, Hata=6.753
demo_samples_mlp_dataset2 klasörü oluşturuluyor...

demo_samples_mlp_dataset2 klasöründeki 8 dosya işleniyor...
Magnitude   : MSE=0.003, MAE=0.049, R²=-9.389
Depth       : MSE=125.330, MAE=8.166, R²=-0.460
Latitude    : MSE=36.938, MAE=4.681, R²=-1.402
Longitu